# recipe-dataclass — worked example 1: Recipe for a unary exp_forward

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `recipe-dataclass`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

A `Recipe` is a 4-field dataclass that captures everything the reverse pass needs to differentiate one node: the forward function, its raw (unboxed) positional args, its keyword args, and a `{argnum: parent_tensor}` dict. A unary op like `exp` has exactly one Tensor input, so it records one parent at argnum 0. The reverse pass reads these fields generically, so every field must be populated even when some are empty.

## Worked solution

We define the `Recipe` dataclass with exactly four fields in the canonical order: `func`, `args`, `kwargs`, `parents`. Then `exp_forward(x)` takes a `MiniTensor`, pulls out its raw `.array`, and computes `torch.exp` on it. We wrap the result in a new `MiniTensor`. The crucial part is attaching a `Recipe`: `func` is `t.exp` (so the dispatcher can look up the matching backward rule), `args` is the 1-tuple `(x.array,)` of the unboxed input (the backward rule replays the call on raw arrays), `kwargs` is `{}` because `exp` takes no keyword arguments, and `parents` is `{0: x}` because positional argument 0 is the Tensor we must keep differentiating into. Dropping any field would break the generic dispatcher.

In [ ]:
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def exp_forward(x: MiniTensor) -> MiniTensor:
    out_arr = t.exp(x.array)
    out = MiniTensor(out_arr)
    out.recipe = Recipe(
        func=t.exp,
        args=(x.array,),     # raw, unboxed input as a 1-tuple
        kwargs={},            # exp takes no kwargs
        parents={0: x},      # arg-0 is the Tensor parent
    )
    return out


x = MiniTensor(t.tensor([0.0, 1.0, 2.0]))
out = exp_forward(x)
print('func is t.exp:', out.recipe.func is t.exp)
print('parents:', out.recipe.parents)
print('out values:', out.array.tolist())